# Features de Calendario
### Festivos, quincenas y eventos del retail mexicano

`Fase 2 · Video 8` — serie **Forecasting de Ventas de la A a la Z**

Arranca la **Fase 2 (feature engineering)**. El calendario es la exógena perfecta: se conoce el futuro
con certeza. Construimos features localizados a México —quincenas, festivos— y **medimos** cuáles
aportan señal. Alimentarán al SARIMAX (Video 13) y al XGBoost (Video 16).

---
## 1. Librerías y carga de datos

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
from sklearn.feature_selection import mutual_info_regression
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
print('✅ Librerías cargadas')

In [ ]:
def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv()
df = pd.read_csv(csv_path, parse_dates=['date']).sort_values('date')

# Serie diaria de demanda total (unidades) y evento comercial por fecha
daily = df.groupby('date')['units_sold'].sum().sort_index()
daily.index = pd.to_datetime(daily.index)
event_by_date = df.groupby('date')['event'].first().reindex(daily.index)

print(f'✅ Serie diaria: {len(daily)} días | {daily.index.min().date()} → {daily.index.max().date()}')

---
## 2. ¿Por qué el calendario es especial?

| Variable exógena | ¿La conoces al pronosticar? |
|---|---|
| Clima futuro | ❌ No (necesitas pronóstico del pronóstico) |
| Precio de la competencia | ❌ Rara vez |
| **Calendario (festivos, quincenas)** | ✅ **Sí, con certeza total** |

Esto hace al calendario ideal como **variable exógena** en modelos como SARIMA**X** (la *X* es de
exógena): puedes extender los features al horizonte futuro sin error, porque las fechas son determinísticas.

**Para aislar la señal de calendario del crecimiento y la estacionalidad anual** (que ya tratamos en los
Videos 3–4), en este notebook trabajamos sobre la demanda **destendenciada**: dividimos cada día entre la
mediana móvil de 31 días. Así lo que queda son los efectos *cortos* — quincena, día de semana, festivos.

In [ ]:
# Destendenciado: aísla los efectos de calendario de corto plazo
trend_31 = daily.rolling(31, center=True, min_periods=1).median()
det = (daily / trend_31).rename('demanda_rel')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(det.index, det.values, color=BLUE, linewidth=0.7, alpha=0.8)
ax.axhline(1.0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Demanda relativa (destendenciada) — sobre esto medimos el calendario',
             fontsize=12, fontweight='bold')
ax.set_ylabel('demanda / mediana móvil 31d')
plt.tight_layout()
plt.show()
print('1.0 = demanda típica de su vecindad; picos = efectos de calendario de corto plazo.')

---
## 3. Construcción de la matriz de features de calendario

Todo se deriva **solo de la fecha** — sin mirar las ventas. Construimos features de tres tipos:

- **De posición temporal:** día de la semana, fin de semana, día del mes, semana del mes.
- **De pago (México):** ventana de quincena, distancia al día de pago más cercano.
- **De eventos:** festivo nacional (con ventana de anticipación) y evento comercial.

Añadimos a propósito un **feature señuelo aleatorio** — sin relación con la demanda — para comprobar más
adelante que nuestro método de selección sabe distinguir señal de ruido.

In [ ]:
idx = daily.index

def days_to_payday(ts):
    cands = (15, ts.days_in_month)
    return min(abs(ts.day - c) for c in cands)

feats = pd.DataFrame(index=idx)
feats['dow']            = idx.dayofweek                      # 0=lun … 6=dom
feats['is_weekend']     = (idx.dayofweek >= 5).astype(int)
feats['day']            = idx.day
feats['week_of_month']  = ((idx.day - 1) // 7 + 1)
feats['is_month_end']   = (idx.day >= idx.days_in_month - 1).astype(int)
feats['days_to_payday'] = [days_to_payday(t) for t in idx]

# Features del calendario mexicano (misma definición que usa el generador)
import sys
sys.path.insert(0, str(csv_path.parents[1]))          # raíz del repo
from src.calendar_mx import quincena_mask, holiday_factor_series

feats['is_quincena']    = quincena_mask(idx).astype(int)
hol_mult, hol_name      = holiday_factor_series(idx)
feats['is_mx_holiday']  = (hol_mult > 1.0).astype(int)
feats['is_event']       = (event_by_date.values != 'Regular').astype(int)

rng = np.random.default_rng(0)
feats['ruido_decoy']    = rng.random(len(idx))

print(f'Matriz de features: {feats.shape[0]} días × {feats.shape[1]} features')
feats.head(8)

---
## 4. La quincena mexicana — el feature estrella

Si graficamos la demanda relativa **por día del mes**, el patrón de pago salta a la vista: repuntes
alrededor del **15** y del **fin/inicio de mes**. Es el pulso quincenal del consumo mexicano.

In [ ]:
by_day = det.groupby(det.index.day).mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
bars_c = [ORANGE if quincena_mask(pd.DatetimeIndex([pd.Timestamp(2022, 1, d)]))[0] else BLUE
          for d in by_day.index]
axes[0].bar(by_day.index, by_day.values, color=bars_c, edgecolor='white')
axes[0].axhline(1.0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('Demanda relativa por día del mes\n(naranja = ventana de quincena)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Día del mes')

q = feats['is_quincena'].astype(bool).values
box = axes[1].boxplot([det.values[~q], det.values[q]], labels=['Normal', 'Quincena'],
                      patch_artist=True, showfliers=False)
for patch, c in zip(box['boxes'], [BLUE, ORANGE]):
    patch.set_facecolor(c); patch.set_alpha(0.6)
axes[1].axhline(1.0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('Demanda relativa: días normales vs quincena', fontsize=12, fontweight='bold')

fig.suptitle('El efecto quincena', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

lift_q = det.values[q].mean() / det.values[~q].mean() - 1
print(f'Lift de quincena (sobre demanda destendenciada): {lift_q:+.1%}')

---
## 5. Festivos nacionales

Los festivos de fecha fija (16 de septiembre, Día de Muertos, Guadalupe…) también mueven la demanda, con
un efecto de **compras de anticipación** en los días previos. Medimos su lift igual que con la quincena.

In [ ]:
h = feats['is_mx_holiday'].astype(bool).values
lift_h = det.values[h].mean() / det.values[~h].mean() - 1

# Perfil promedio alrededor de un festivo grande (Independencia, 16 sep)
win = range(-6, 4)
prof = []
for off in win:
    dts = [pd.Timestamp(y, 9, 16) + pd.Timedelta(days=off) for y in range(2021, 2025)]
    vals = [det.get(pd.Timestamp(d), np.nan) for d in dts]
    prof.append(np.nanmean(vals))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].bar(['Sin festivo', 'Festivo (+ventana)'],
            [det.values[~h].mean(), det.values[h].mean()],
            color=[BLUE, GREEN], edgecolor='white', width=0.5)
axes[0].axhline(1.0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title(f'Demanda relativa — lift festivo: {lift_h:+.1%}', fontsize=12, fontweight='bold')

axes[1].plot(list(win), prof, marker='o', color=GREEN, linewidth=2)
axes[1].axvline(0, color=RED, linestyle='--', linewidth=1, label='16 sep')
axes[1].axhline(1.0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('Perfil alrededor de Independencia (promedio 2021–2024)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Días respecto al festivo'); axes[1].legend()

fig.suptitle('El efecto de los festivos nacionales', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Lift de festivos (con ventana de anticipación): {lift_h:+.1%}')

---
## 6. ¿Qué features realmente aportan? Información mutua

Construir features es fácil; el error es quedarse con todos. Medimos la **información mutua** (MI) entre
cada feature y la demanda relativa: cuánta incertidumbre sobre la demanda elimina conocer el feature. A
diferencia de la correlación, la MI captura relaciones **no lineales** y no direccionales.

La prueba de fuego: nuestro **feature señuelo** debe quedar hasta el fondo del ranking.

In [ ]:
X = feats.copy()
y = det.values
discrete = [c for c in X.columns if c not in ('days_to_payday', 'ruido_decoy')]
mi = mutual_info_regression(
    X.values, y,
    discrete_features=[X.columns.get_loc(c) for c in discrete],
    random_state=0,
)
mi_s = pd.Series(mi, index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(11, 6))
colors = [RED if f == 'ruido_decoy' else GREEN for f in mi_s.index]
ax.barh(mi_s.index, mi_s.values, color=colors, edgecolor='white')
ax.set_title('Información mutua feature ↔ demanda (rojo = señuelo)', fontsize=13, fontweight='bold')
ax.set_xlabel('MI (bits, aprox.)')
plt.tight_layout()
plt.show()

print('Ranking de features por información mutua:')
for f, v in mi_s.sort_values(ascending=False).items():
    flag = '  ← señuelo (debe ser ~0)' if f == 'ruido_decoy' else ''
    print(f'  {f:<16}: {v:.4f}{flag}')

print('\n⚠️  Matiz clave: `is_mx_holiday` queda casi al fondo, pero NO porque el efecto sea')
print('   débil — su lift (sección 5) es real. Es porque es un feature RARO (pocos días al')
print('   año) y la MI mide información sobre TODA la serie. Frecuencia ≠ tamaño del efecto:')
print('   por eso se lee la MI JUNTO con el lift, no en su lugar. El señuelo, en cambio, queda')
print('   en ~0 tanto en MI como en lift → ese sí es ruido puro.')

---
## 7. La revelación: el calendario que inyectamos

Como generamos el dataset, podemos comparar el efecto **medido** contra el que **inyectamos** en
`src/calendar_mx.py`. Es la validación de que nuestros features capturan algo real y no un artefacto.

In [ ]:
from src.calendar_mx import QUINCENA_STRENGTH, MX_HOLIDAYS, HOLIDAY_PRE_WINDOW

print(f'Quincena inyectada: +{QUINCENA_STRENGTH:.0%}  |  medido (destendenciado): {lift_q:+.1%}')
print('(el medido supera al inyectado porque los días normales quedan por debajo de la')
print(' mediana móvil, que ya incorpora el empujón quincenal — comparación contra el valle)\n')

print(f'Festivos inyectados (ventana pre-festivo = {HOLIDAY_PRE_WINDOW} días):')
for hh in sorted(MX_HOLIDAYS, key=lambda x: x.mult, reverse=True):
    print(f'  {hh.name:<24} {hh.month:>2}/{hh.day:<2}  ×{hh.mult}')

print('\nModerado: la señal medida coincide en dirección y orden de magnitud con el diseño.')

---
## 8. Buenas prácticas

### Encoding cíclico
El día de la semana y el mes son **cíclicos**: diciembre (12) está tan cerca de enero (1) como de
noviembre (11), pero como números no lo parecen. Para modelos que asumen distancia (lineales, redes) se
codifican con seno/coseno:

$$x_{\sin} = \sin\!\left(\frac{2\pi\, t}{T}\right), \quad x_{\cos} = \cos\!\left(\frac{2\pi\, t}{T}\right)$$

Los modelos de árboles (XGBoost) no lo necesitan: parten por umbrales y manejan bien el entero.

### Fuga de datos (data leakage)
Los features de calendario son **seguros**: se conocen de antemano, así que puedes extenderlos al futuro
sin mirar el pasado. Ojo — esto **no** aplica a los lags y ventanas móviles del próximo video, donde el
leakage sí es un riesgo real.

### Cómo se usan por tier (Video 7)
- **Tier A (SARIMAX):** entran como matriz **exógena** `X` (quincena, festivo, evento).
- **Tier B (XGBoost):** son columnas más del dataset tabular.
- **Tier C (intermitente):** se usan con cuidado; con tanta demanda cero, el calendario aporta poco.

In [ ]:
enc = pd.DataFrame(index=daily.index)
enc['dow_sin']   = np.sin(2*np.pi*daily.index.dayofweek/7)
enc['dow_cos']   = np.cos(2*np.pi*daily.index.dayofweek/7)
enc['month_sin'] = np.sin(2*np.pi*(daily.index.month-1)/12)
enc['month_cos'] = np.cos(2*np.pi*(daily.index.month-1)/12)

fig, ax = plt.subplots(figsize=(5, 5))
u = enc.drop_duplicates('dow_sin')
ax.scatter(u['dow_sin'], u['dow_cos'], s=120, color=PURPLE, zorder=3)
for _, r in u.iterrows():
    d = int(round(np.arctan2(r['dow_sin'], r['dow_cos'])/(2*np.pi)*7)) % 7
    ax.annotate(['L','M','X','J','V','S','D'][d], (r['dow_sin'], r['dow_cos']),
                ha='center', va='center', fontweight='bold', color='white')
ax.set_title('Día de la semana en el círculo\n(sin/cos: dom queda junto a lun)', fontsize=11, fontweight='bold')
ax.set_xlabel('dow_sin'); ax.set_ylabel('dow_cos'); ax.set_aspect('equal')
plt.tight_layout()
plt.show()
print('Con sin/cos, la distancia entre domingo y lunes deja de ser 6 y vuelve a ser 1.')

---
## 9. Conclusiones y puente al Video 9

### Las reglas que te llevas

1. **El calendario es la exógena perfecta:** determinística y conocida al futuro. Explótala.
2. **Localiza al mercado:** en México, la **quincena** y los festivos nacionales mueven la aguja — un
   diferenciador frente a features genéricos.
3. **Construir ≠ servir.** Evalúa cada feature con MI **y** tamaño de efecto (lift): descarta el ruido,
   pero no descartes un feature raro-pero-fuerte (como los festivos) solo por su baja MI global.
4. **Codifica lo cíclico** con sin/cos para modelos que asumen distancia; los árboles no lo necesitan.
5. **El calendario no tiene fuga de datos** — pero lo que viene (lags) sí. Cuidado.

### El mapa

Estos features son exógenas de calidad para el **tier A (SARIMAX)** y columnas del **tier B (XGBoost)**.
Pero un modelo temporal necesita también **memoria**: qué pasó ayer, la semana pasada, el mes pasado.

---

### Próximo video
**Video 9 — Lags y ventanas móviles: el alma del feature engineering temporal**
Construiremos lags, medias/desviaciones móviles, expanding y EWM — y veremos el error clásico que
introduce **fuga de datos**: usar ventanas centradas que miran el futuro.